# KPI Availability Batch — QC Notebook

Steps through every stage of the batch one cell at a time so outputs can be inspected.

| Step | What it does |
|------|--------------|
| 0 | Imports & config |
| 1 | Snowflake session |
| 2 | Load raw OSA data |
| 3 | Load sites & feedback |
| 4 | Store-level OSA scores (TW / LW / Month / Year) |
| 5 | Cluster-level OSA |
| 6 | Department-level OSA |
| 7 | Time-of-day OSA |
| 8 | Feedback metrics |
| 9 | Join store scores |
| 10 | Assemble per-store JSON (single store preview) |
| 11 | Cortex key insights (single store preview) |
| 12 | Write to Snowflake |


## Step 0 — Imports & config

In [ ]:
import json
import os
import sys
from datetime import date, timedelta
from pathlib import Path

from dotenv import load_dotenv
from snowflake.snowpark import Session
from snowflake.snowpark import functions as F
from snowflake.snowpark.types import (
    StructType, StructField,
    StringType, IntegerType, FloatType,
)

# ── Load credentials ────────────────────────────────────────────────────────
load_dotenv(dotenv_path=Path(".") / ".env")

# ── Date config ─────────────────────────────────────────────────────────────
# Set ANALYSIS_DATE to the last day of available OSA data (typically yesterday)
ANALYSIS_DATE_STR = '2026-03-20'
RUN_DATE_STR      = str(date.today())

# ── Optional: restrict to a single store for fast QC ────────────────────────
QC_STORE_ID = None   # e.g. 436  — set to None to process all stores

# ── Write config ────────────────────────────────────────────────────────────
OVERWRITE = False   # True = overwrite output table for this date

# ── Constants ───────────────────────────────────────────────────────────────
OSA_TABLE       = 'ANALYSIS.FINAL.LSM_REPORT_ITEM_STORE_DAY_REPORTING_PERIOD'
HIERARCHY_TABLE = 'ANALYSIS.FINAL.LSM_ITEM_HIERARCHY'
SITE_TABLE      = 'ANALYSIS.FINAL.LSM_SITE_HIERARCHY'
FEEDBACK_TABLE  = 'ANALYSIS.FINAL.LSM_ALERT_FEEDBACK'
OUTPUT_TABLE    = 'SNOWFLAKE_LEARNING_DB.STORE_INSIGHT.KPI_AVAILABILITY_SUMMARY'

OSA_TARGET    = 0.90
PERIOD_LABELS = {1: 'Morning (6am-12pm)', 2: 'Afternoon (12pm-6pm)', 3: 'Evening (6pm-Close)'}
DAY_ORDER     = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']

# ── Derive date windows ─────────────────────────────────────────────────────
analysis_date    = date.fromisoformat(ANALYSIS_DATE_STR)
TW_END           = str(analysis_date)
TW_START         = str(analysis_date - timedelta(days=6))
LW_END           = str(analysis_date - timedelta(days=7))
LW_START         = str(analysis_date - timedelta(days=13))
MONTH_START      = str(analysis_date - timedelta(days=27))
YEAR_START       = str(analysis_date - timedelta(days=364))
LOOKBACK         = YEAR_START

print(f'Analysis date : {ANALYSIS_DATE_STR}')
print(f'Run date      : {RUN_DATE_STR}')
print(f'This week     : {TW_START} – {TW_END}')
print(f'Last week     : {LW_START} – {LW_END}')
print(f'QC store      : {QC_STORE_ID}')

: 

## Step 1 — Snowflake session

In [ ]:
params = {
    'account':   os.environ['SNOWFLAKE_ACCOUNT'],
    'user':      os.environ['SNOWFLAKE_USER'],
    'role':      os.getenv('SNOWFLAKE_ROLE',      'ACL_SNOWFLAKE_RI_EOE_PRD_DP_USER'),
    'warehouse': os.getenv('SNOWFLAKE_WAREHOUSE', 'SNOWFLAKE_LEARNING_WH'),
    'database':  os.getenv('SNOWFLAKE_DATABASE',  'SNOWFLAKE_LEARNING_DB'),
    'schema':    os.getenv('SNOWFLAKE_SCHEMA',     'STORE_INSIGHT'),
}
password = os.getenv('SNOWFLAKE_PASSWORD')
if password:
    params['password'] = password
else:
    params['authenticator'] = 'externalbrowser'

session = Session.builder.configs(params).create()
print('Session created.')

: 

## Step 2 — Load raw OSA data (joined to item hierarchy)

In [15]:
hier = (
    session.table(HIERARCHY_TABLE)
    .select(
        F.col('MASTER_ITEM_ID').alias('ITEM_CODE'),
        F.col('DEPARTMENT_NAME'),
        F.col('PRODUCT_PROFILE_GROUP_NAME'),
    )
    .distinct()
)

raw = (
    session.table(OSA_TABLE)
    .filter(F.col('ANALYSIS_DATE') >= LOOKBACK)
    .filter(F.col('ANALYSIS_DATE') <= TW_END)
    .select(
        F.col('SITE_ID').alias('STORE_ID'),
        F.col('ITEM_ID').alias('ITEM_CODE'),
        F.col('ANALYSIS_DATE'),
        F.col('REPORTING_PERIOD_ID'),
        F.col('ACTUAL_SALES_VOLUME'),
        F.col('LOST_SALES_VOLUME'),
        F.col('LOST_SALES_VALUE'),
        F.dayname(F.col('ANALYSIS_DATE')).alias('DAY_NAME'),
    )
)

osa = raw.join(hier, on=['ITEM_CODE'], how='left').cache_result()

# Optional: restrict to a single store for QC
if QC_STORE_ID:
    osa = osa.filter(F.col('STORE_ID') == QC_STORE_ID).cache_result()

print(f'OSA rows loaded: {osa.count():,}')
osa.show(3)

OSA rows loaded: 389,236,544
---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|"ITEM_CODE"  |"STORE_ID"  |"ANALYSIS_DATE"  |"REPORTING_PERIOD_ID"  |"ACTUAL_SALES_VOLUME"  |"LOST_SALES_VOLUME"  |"LOST_SALES_VALUE"  |"DAY_NAME"  |"DEPARTMENT_NAME"  |"PRODUCT_PROFILE_GROUP_NAME"  |
---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|818422       |204         |2025-03-21       |1                      |0.0000                 |0.0000               |0E-8                |Fri         |READY MEALS        |SNACKS - BURGER               |
|15888        |204         |2025-03-21       |1                      |0.0000                 |0.0000               |0E-8                |Fri         |STATIONERY   

## Step 3 — Load sites & feedback

In [16]:
sites = (
    session.table(SITE_TABLE)
    .select(F.col('SITE_ID').alias('STORE_ID'), F.col('CLUSTER_ID'))
    .distinct()
)
print(f'Sites: {sites.count():,}')
sites.show(3)

feedback = (
    session.table(FEEDBACK_TABLE)
    .filter(F.col('ALERT_ANALYSIS_DATE') >= TW_START)
    .filter(F.col('ALERT_ANALYSIS_DATE') <= TW_END)
    .select(
        F.col('SITE_ID').alias('STORE_ID'),
        F.col('TICK_AND_FLICK_FLAG'),
        F.col('INTERVENTION_NAME'),
        F.col('ALERT_ANALYSIS_DATE'),
    )
    .cache_result()
)
print(f'Feedback rows: {feedback.count():,}')
feedback.show(3)

Sites: 174
-----------------------------
|"STORE_ID"  |"CLUSTER_ID"  |
-----------------------------
|205         |6             |
|454         |4             |
|512         |2             |
-----------------------------

Feedback rows: 9,337
------------------------------------------------------------------------------------
|"STORE_ID"  |"TICK_AND_FLICK_FLAG"  |"INTERVENTION_NAME"  |"ALERT_ANALYSIS_DATE"  |
------------------------------------------------------------------------------------
|444         |0                      |NULL                 |2026-03-19             |
|765         |0                      |NULL                 |2026-03-19             |
|641         |0                      |NULL                 |2026-03-20             |
------------------------------------------------------------------------------------



## Step 4 — Store-level OSA scores (TW / LW / Month / Year)

In [17]:
def _osa_pct(sales_col, lsv_col):
    denom = sales_col + lsv_col
    return F.when(denom > 0, sales_col / denom).otherwise(F.lit(0))

def _store_osa(osa, start, end):
    return (
        osa.filter((F.col('ANALYSIS_DATE') >= start) & (F.col('ANALYSIS_DATE') <= end))
        .group_by('STORE_ID')
        .agg(
            F.sum('ACTUAL_SALES_VOLUME').alias('SALES'),
            F.sum('LOST_SALES_VOLUME').alias('LSV'),
            F.sum('LOST_SALES_VALUE').alias('LOST_SALES_VALUE'),
        )
        .with_column('OSA', _osa_pct(F.col('SALES'), F.col('LSV')))
    )

tw_store = _store_osa(osa, TW_START, TW_END)
lw_store = _store_osa(osa, LW_START, LW_END)
m_store  = _store_osa(osa, MONTH_START, TW_END)
y_store  = _store_osa(osa, YEAR_START,  TW_END)

print('This week store OSA (sample):')
tw_store.sort(F.col('OSA').asc()).show(5)

print('Last week store OSA (sample):')
lw_store.sort(F.col('OSA').asc()).show(5)

This week store OSA (sample):
---------------------------------------------------------------------------
|"STORE_ID"  |"SALES"     |"LSV"      |"LOST_SALES_VALUE"  |"OSA"         |
---------------------------------------------------------------------------
|204         |1.0000      |11.8238    |21.67000000         |0.0779800059  |
|759         |3328.0000   |740.8203   |1332.42000000       |0.8179274961  |
|535         |29360.0000  |4019.8916  |8212.92000000       |0.8795714603  |
|762         |2577.0000   |344.1977   |769.80000000        |0.8821724048  |
|761         |2153.0000   |253.4022   |624.03000000        |0.8946966554  |
---------------------------------------------------------------------------

Last week store OSA (sample):
---------------------------------------------------------------------------
|"STORE_ID"  |"SALES"     |"LSV"      |"LOST_SALES_VALUE"  |"OSA"         |
---------------------------------------------------------------------------
|204         |11306.0000  |

## Step 5 — Cluster-level OSA

In [18]:
def _cluster_osa(osa, sites, start, end):
    store_scores = _store_osa(osa, start, end).join(sites, on=['STORE_ID'], how='left')
    return (
        store_scores
        .group_by('CLUSTER_ID')
        .agg(
            F.sum('SALES').alias('CLUSTER_SALES'),
            F.sum('LSV').alias('CLUSTER_LSV'),
        )
        .with_column('CLUSTER_OSA', _osa_pct(F.col('CLUSTER_SALES'), F.col('CLUSTER_LSV')))
    )

tw_cluster = _cluster_osa(osa, sites, TW_START, TW_END)
print(f'Clusters: {tw_cluster.count()}')
tw_cluster.show(10)

Clusters: 6
------------------------------------------------------------------
|"CLUSTER_ID"  |"CLUSTER_SALES"  |"CLUSTER_LSV"  |"CLUSTER_OSA"  |
------------------------------------------------------------------
|4             |330310.0000      |21663.0594     |0.9384525070   |
|3             |378685.0000      |26354.7977     |0.9349328193   |
|1             |298947.0000      |20234.8333     |0.9366040570   |
|6             |502072.7940      |41203.1631     |0.9241579485   |
|2             |530807.0000      |32870.8773     |0.9416849967   |
|5             |366546.0000      |23770.9465     |0.9390983489   |
------------------------------------------------------------------



## Step 6 — Department-level OSA

In [19]:
def _dept_osa(osa, start, end):
    return (
        osa.filter((F.col('ANALYSIS_DATE') >= start) & (F.col('ANALYSIS_DATE') <= end))
        .filter(F.col('DEPARTMENT_NAME').is_not_null())
        .group_by('STORE_ID', 'DEPARTMENT_NAME')
        .agg(
            F.sum('ACTUAL_SALES_VOLUME').alias('DEPT_SALES'),
            F.sum('LOST_SALES_VOLUME').alias('DEPT_LSV'),
            F.sum('LOST_SALES_VALUE').alias('DEPT_LSV_VALUE'),
        )
        .with_column('DEPT_OSA', _osa_pct(F.col('DEPT_SALES'), F.col('DEPT_LSV')))
    )

tw_dept = _dept_osa(osa, TW_START, TW_END)
lw_dept = _dept_osa(osa, LW_START, LW_END)

print('This week dept OSA (worst 5):')
tw_dept.filter(F.col('DEPT_SALES') + F.col('DEPT_LSV') > 0).sort(F.col('DEPT_OSA').asc()).show(5)

This week dept OSA (worst 5):
----------------------------------------------------------------------------------------------
|"STORE_ID"  |"DEPARTMENT_NAME"  |"DEPT_SALES"  |"DEPT_LSV"  |"DEPT_LSV_VALUE"  |"DEPT_OSA"  |
----------------------------------------------------------------------------------------------
|463         |ETHNIC             |0.0000        |1.0390      |0E-8              |0E-10       |
|432         |TOYS/SPORTS        |0.0000        |1.1896      |1.00000000        |0E-10       |
|419         |TOYS/SPORTS        |0.0000        |0.2678      |0E-8              |0E-10       |
|490         |TOYS/SPORTS        |0.0000        |0.6639      |0E-8              |0E-10       |
|453         |TOYS/SPORTS        |0.0000        |0.2560      |0.32000000        |0E-10       |
----------------------------------------------------------------------------------------------



## Step 7 — Time-of-day OSA

In [20]:
def _tod_osa(osa, start, end):
    return (
        osa.filter((F.col('ANALYSIS_DATE') >= start) & (F.col('ANALYSIS_DATE') <= end))
        .filter(F.col('REPORTING_PERIOD_ID').isin(1, 2, 3))
        .group_by('STORE_ID', 'DAY_NAME', 'REPORTING_PERIOD_ID')
        .agg(
            F.sum('ACTUAL_SALES_VOLUME').alias('TOD_SALES'),
            F.sum('LOST_SALES_VOLUME').alias('TOD_LSV'),
        )
        .with_column('TOD_OSA', _osa_pct(F.col('TOD_SALES'), F.col('TOD_LSV')))
    )

tod_data = _tod_osa(osa, TW_START, TW_END)
print(f'Time-of-day rows: {tod_data.count():,}')
tod_data.sort('STORE_ID', 'DAY_NAME', 'REPORTING_PERIOD_ID').show(9)

Time-of-day rows: 2,519
--------------------------------------------------------------------------------------------
|"STORE_ID"  |"DAY_NAME"  |"REPORTING_PERIOD_ID"  |"TOD_SALES"  |"TOD_LSV"  |"TOD_OSA"     |
--------------------------------------------------------------------------------------------
|204         |Fri         |2                      |1.0000       |11.8238    |0.0779800059  |
|205         |Fri         |1                      |1329.0000    |168.5615   |0.8874426860  |
|205         |Fri         |2                      |1942.0000    |160.2147   |0.9237876607  |
|205         |Fri         |3                      |1272.0000    |109.0177   |0.9210598821  |
|205         |Mon         |1                      |1236.0000    |126.3910   |0.9072285416  |
|205         |Mon         |2                      |1576.0000    |137.0289   |0.9200078294  |
|205         |Mon         |3                      |1004.0000    |125.1572   |0.8891587460  |
|205         |Sat         |1                  

## Step 8 — Feedback metrics (actionability, compliance, tick & flick)

In [11]:
def _feedback_metrics(feedback):
    total  = feedback.group_by('STORE_ID').agg(F.count('*').alias('TOTAL_ALERTS'))
    worked = (
        feedback.filter(F.col('INTERVENTION_NAME').is_not_null())
        .group_by('STORE_ID').agg(F.count('*').alias('WORKED_ALERTS'))
    )
    taf = (
        feedback.filter(F.col('TICK_AND_FLICK_FLAG') == True)
        .group_by('STORE_ID').agg(F.count('*').alias('TAF_ALERTS'))
    )
    return (
        total
        .join(worked, on=['STORE_ID'], how='left')
        .join(taf,    on=['STORE_ID'], how='left')
        .with_column('WORKED_ALERTS', F.coalesce(F.col('WORKED_ALERTS'), F.lit(0)))
        .with_column('TAF_ALERTS',    F.coalesce(F.col('TAF_ALERTS'),    F.lit(0)))
        .with_column('ACTIONABILITY',    F.col('WORKED_ALERTS') / F.col('TOTAL_ALERTS'))
        .with_column('ALERT_COMPLIANCE', F.col('WORKED_ALERTS') / F.col('TOTAL_ALERTS'))
        .with_column('TICK_FLICK_RATE',  F.col('TAF_ALERTS')    / F.col('TOTAL_ALERTS'))
    )

fb_metrics = _feedback_metrics(feedback)
print(f'Feedback metric rows: {fb_metrics.count()}')
fb_metrics.show(5)

Feedback metric rows: 121
---------------------------------------------------------------------------------------------------------------------------
|"STORE_ID"  |"TOTAL_ALERTS"  |"WORKED_ALERTS"  |"TAF_ALERTS"  |"ACTIONABILITY"  |"ALERT_COMPLIANCE"  |"TICK_FLICK_RATE"  |
---------------------------------------------------------------------------------------------------------------------------
|484         |153             |59               |33            |0.385621         |0.385621            |0.215686           |
|759         |50              |0                |0             |0.000000         |0.000000            |0.000000           |
|453         |36              |36               |8             |1.000000         |1.000000            |0.222222           |
|494         |51              |51               |24            |1.000000         |1.000000            |0.470588           |
|215         |91              |91               |16            |1.000000         |1.000000            |0.1

## Step 9 — Join all store scores together

In [21]:
store_joined = (
    tw_store.select(F.col('STORE_ID'), F.col('OSA').alias('TW_OSA'))
    .join(lw_store.select(F.col('STORE_ID'), F.col('OSA').alias('LW_OSA')), on=['STORE_ID'], how='left')
    .join(m_store.select( F.col('STORE_ID'), F.col('OSA').alias('M_OSA')),  on=['STORE_ID'], how='left')
    .join(y_store.select( F.col('STORE_ID'), F.col('OSA').alias('Y_OSA')),  on=['STORE_ID'], how='left')
    .join(sites,                                                              on=['STORE_ID'], how='left')
    .join(tw_cluster.select(F.col('CLUSTER_ID'), F.col('CLUSTER_OSA')),      on=['CLUSTER_ID'], how='left')
    .join(fb_metrics.select(
        F.col('STORE_ID'),
        F.col('ACTIONABILITY'), F.col('ALERT_COMPLIANCE'), F.col('TICK_FLICK_RATE')
    ), on=['STORE_ID'], how='left')
)

store_rows = store_joined.collect()
print(f'Stores to process: {len(store_rows)}')
store_joined.show(5)

Stores to process: 122
--------------------------------------------------------------------------------------------------------------------------------------------------------------------
|"STORE_ID"  |"CLUSTER_ID"  |"TW_OSA"      |"LW_OSA"      |"M_OSA"       |"Y_OSA"       |"CLUSTER_OSA"  |"ACTIONABILITY"  |"ALERT_COMPLIANCE"  |"TICK_FLICK_RATE"  |
--------------------------------------------------------------------------------------------------------------------------------------------------------------------
|638         |6             |0.9112808304  |0.9108345239  |0.9203685606  |0.9084985719  |0.9241579485   |1.000000         |1.000000            |0.490196           |
|759         |4             |0.8179274961  |0.8190263737  |0.8358462442  |0.8235296402  |0.9384525070   |0.000000         |0.000000            |0.000000           |
|209         |6             |0.9379233295  |0.9327318460  |0.9332439764  |0.9125416119  |0.9241579485   |1.000000         |1.000000            |0.213483

## Step 10 — Assemble per-store JSON (preview first store)

In [22]:
# Collect dept & tod data indexed by store
dept_tw_rows = tw_dept.collect()
dept_lw_rows = lw_dept.collect()
tod_rows     = tod_data.collect()

dept_tw_by_store = {}
for r in dept_tw_rows:
    dept_tw_by_store.setdefault(int(r['STORE_ID']), []).append(r)

dept_lw_by_store = {}
for r in dept_lw_rows:
    dept_lw_by_store.setdefault(int(r['STORE_ID']), []).append(r)

tod_by_store = {}
for r in tod_rows:
    tod_by_store.setdefault(int(r['STORE_ID']), []).append(r)

# Preview just the first store
row = store_rows[0]
sid = int(row['STORE_ID'])

tw_osa = float(row['TW_OSA'] or 0)
lw_osa = float(row['LW_OSA'] or 0)
m_osa  = float(row['M_OSA']  or 0)
y_osa  = float(row['Y_OSA']  or 0)
cl_osa = float(row['CLUSTER_OSA'] or 0)
wow_pp = (tw_osa - lw_osa) * 100

actionability    = float(row['ACTIONABILITY']    or 0) * 100
alert_compliance = float(row['ALERT_COMPLIANCE'] or 0) * 100
tick_flick       = float(row['TICK_FLICK_RATE']  or 0) * 100

dept_tw = {r['DEPARTMENT_NAME']: r for r in dept_tw_by_store.get(sid, [])}
dept_lw = {r['DEPARTMENT_NAME']: r for r in dept_lw_by_store.get(sid, [])}

category_perf = []
for dname, dr in sorted(dept_tw.items()):
    dept_osa_val = float(dr['DEPT_OSA'] or 0) * 100
    category_perf.append({
        'category':    dname,
        'performance': f'{dept_osa_val:.1f}%',
        'warning':     dept_osa_val < OSA_TARGET * 100,
    })

sorted_depts = sorted(dept_tw.values(), key=lambda r: float(r['DEPT_LSV_VALUE'] or 0), reverse=True)
top_drivers = []
for dr in sorted_depts[:4]:
    dname    = dr['DEPARTMENT_NAME']
    d_tw_osa = float(dr['DEPT_OSA'] or 0) * 100
    d_lw_row = dept_lw.get(dname)
    d_lw_osa = float(d_lw_row['DEPT_OSA'] or 0) * 100 if d_lw_row else d_tw_osa
    d_delta  = d_tw_osa - d_lw_osa
    top_drivers.append({
        'driver':          dname,
        'Subcategory':     '',
        'When':            '',
        'trend_direction': 'up' if d_delta >= 0 else 'down',
        'trend':           f'{d_delta:+.1f}pp vs LW',
    })

tod_store  = tod_by_store.get(sid, [])
tod_by_day = {}
for tr in tod_store:
    tod_by_day.setdefault(tr['DAY_NAME'], {})[int(tr['REPORTING_PERIOD_ID'])] = float(tr['TOD_OSA'] or 0)

time_of_day = []
for day_name in DAY_ORDER:
    periods = tod_by_day.get(day_name)
    if not periods:
        continue
    day_periods = []
    for pid in [1, 2, 3]:
        if pid in periods:
            pct = periods[pid] * 100
            day_periods.append({
                'time_period': PERIOD_LABELS[pid],
                'performance': f'{pct:.1f}%',
                'rag_status':  'green' if pct >= OSA_TARGET * 100 else 'red',
            })
    if day_periods:
        time_of_day.append({'day': day_name, 'performance': day_periods})

direction = 'increased' if wow_pp >= 0 else 'decreased'
summary   = (
    f'Availability {direction} {abs(wow_pp):.1f}pp this week vs last week '
    f'(store OSA: {tw_osa*100:.1f}%, cluster average: {cl_osa*100:.1f}%).'
)

print(f'Store {sid}')
print(f'  TW OSA:  {tw_osa*100:.1f}%  LW OSA: {lw_osa*100:.1f}%  WoW: {wow_pp:+.1f}pp')
print(f'  Cluster: {cl_osa*100:.1f}%')
print(f'  Feedback — actionability: {actionability:.1f}%  compliance: {alert_compliance:.1f}%  t&f: {tick_flick:.1f}%')
print(f'  Summary: {summary}')
print(f'  Top drivers: {[d["driver"] for d in top_drivers]}')
print(f'  Category perf (first 5): {category_perf[:5]}')
print(f'  ToD days with data: {[d["day"] for d in time_of_day]}')

Store 242
  TW OSA:  92.4%  LW OSA: 93.2%  WoW: -0.8pp
  Cluster: 93.9%
  Feedback — actionability: 89.4%  compliance: 89.4%  t&f: 25.5%
  Summary: Availability decreased 0.8pp this week vs last week (store OSA: 92.4%, cluster average: 93.9%).
  Top drivers: ['CIGARETTES', 'SOFT DRINKS', 'CONFECTIONERY', 'IN-STORE BAKERY']
  Category perf (first 5): [{'category': 'ACCESSORIES', 'performance': '93.5%', 'warning': False}, {'category': 'ACCESSORIES/FASHION', 'performance': '0.0%', 'warning': True}, {'category': 'AFFORDABLE/LOW ALC WINES', 'performance': '75.7%', 'warning': True}, {'category': 'AMBIENT - READY MEAL', 'performance': '89.8%', 'warning': True}, {'category': 'BABY - TOILETRIES', 'performance': '99.1%', 'warning': False}]
  ToD days with data: ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']


## Step 11 — Cortex key insights (preview first store)

In [23]:
underperforming = [cp['category'] for cp in category_perf if cp['warning']][:3]

facts = {
    'current_week_osa_pct':  tw_osa * 100,
    'last_week_osa_pct':     lw_osa * 100,
    'wow_change_pp':         wow_pp,
    'cluster_osa_pct':       cl_osa * 100,
    'month_avg_osa_pct':     m_osa  * 100,
    'year_avg_osa_pct':      y_osa  * 100,
    'top_underperforming_depts': ', '.join(underperforming) if underperforming else 'none',
}

prompt = (
    'You are a retail performance analyst. '
    'Based on the store KPI data below, generate exactly 3 concise insight sentences '
    'for a store manager. '
    'You MUST respond with a valid JSON array of exactly 3 strings and nothing else. '
    'Example format: ["Insight one.", "Insight two.", "Insight three."] '
    'Do not include any text outside the JSON array.\n\n'
    'Store KPI data:\n'
    f'- Current week OSA: {facts["current_week_osa_pct"]:.1f}%\n'
    f'- Last week OSA: {facts["last_week_osa_pct"]:.1f}%\n'
    f'- Week-on-week change: {facts["wow_change_pp"]:+.1f}pp\n'
    f'- Cluster average OSA: {facts["cluster_osa_pct"]:.1f}%\n'
    f'- Rolling 28-day OSA: {facts["month_avg_osa_pct"]:.1f}%\n'
    f'- Rolling 365-day OSA: {facts["year_avg_osa_pct"]:.1f}%\n'
    f'- Availability target: 90%\n'
    f'- Top underperforming departments: {facts["top_underperforming_depts"]}\n'
)

escaped = prompt.replace("'", "\\'")
result  = session.sql(
    f"SELECT SNOWFLAKE.CORTEX.COMPLETE('mistral-large', '{escaped}') AS RESPONSE"
).collect()
raw = result[0]['RESPONSE'].strip()
print('Raw Cortex response:')
print(raw)
print()

if raw.startswith('```'):
    raw = raw.split('```')[1]
    if raw.startswith('json'):
        raw = raw[4:]
    raw = raw.strip()

insights = json.loads(raw)
print('Parsed insights:')
for i, ins in enumerate(insights, 1):
    print(f'  {i}. {ins}')

Raw Cortex response:
[
  "The current week's OSA of 92.4% is below both the last week's OSA of 93.2% and the cluster average of 93.9%, indicating a decrease in performance.",
  "Despite the week-on-week decrease, the rolling 28-day OSA is still above the availability target of 90% at 93.2%.",
  "The top underperforming departments are ACCESSORIES/FASHION, AFFORDABLE/LOW ALC WINES, and AMBIENT - READY MEAL, suggesting these areas need focused attention."
]

Parsed insights:
  1. The current week's OSA of 92.4% is below both the last week's OSA of 93.2% and the cluster average of 93.9%, indicating a decrease in performance.
  2. Despite the week-on-week decrease, the rolling 28-day OSA is still above the availability target of 90% at 93.2%.
  3. The top underperforming departments are ACCESSORIES/FASHION, AFFORDABLE/LOW ALC WINES, and AMBIENT - READY MEAL, suggesting these areas need focused attention.


## Step 12 — Write to Snowflake

Only run this cell when you're happy with the outputs above.

In [24]:
import sys
sys.path.insert(0, str(Path('.')))
from kpi_availability_batch import run_batch

run_batch(ANALYSIS_DATE_STR, run_date_str=RUN_DATE_STR, overwrite=OVERWRITE)

KPI Availability Batch — analysis date: 2026-03-20  |  run date: 2026-03-23
  This week : 2026-03-14 – 2026-03-20
  Last week : 2026-03-07 – 2026-03-13


 pip install snowflake-connector-python[secure-local-storage]


Initiating login request with your identity provider. A browser window should have opened for you to complete the login. If you can't see it, check existing browser windows, or your OS settings. Press CTRL+C to abort and try again...
Going to open: https://login.microsoftonline.com/93a31d5d-1a5d-433f-baa0-138549cd2108/saml2?SAMLRequest=nZLBctowEIZfxaOebUu2YUCDybihmdCmlAGTpLkJS4CCLRmtHJM%2BfYUJM%2BkhOfTi8cjfrj%2Ftv6OrY1V6L8KA1CpFJMDIE6rQXKptilb5jT9AHlimOCu1Eil6FYCuxiNgVVnTrLE7tRCHRoD1XCMFtPuQosYoqhlIoIpVAqgt6DL7eUejANPaaKsLXaJ3JZ9XMABhrDO8lHCQTm9nbU3DsG3boI0DbbZhhDEO8TB01An5cuGP7k4f8CTEyYl3hMPnb25fpTqP4DOt9RkCepvnc3%2F%2Ba5kjL7uoXmsFTSXMUpgXWYjV4u4sAM4AKtKPcRI0ex%2B0m2HA%2FjRGBKB0uynZXhS6qhvrWgfuLdwIHpZ6K93tp5MU1XvJZ0nO%2Bk%2F3h4fdt2f547le17f5Yv34MMt%2B94g97r%2Fnq8fhtinWh6xA3v0l3ugU7xSgEVN1CtW6Ixz1fRz7UZwTQpMhjaOgT8gT8iYuVKmY7Sov5p1HUMnCaNAbq1UplegshzGLCe9xnzD3SOJ4468Zwz6JB71kWPCI4EF4ijpC5%2FWhnYgZ%2F9dQRuH7Fm%2FrOHMJTSdzXcri1bvRpmL24wBJQLoTyf1Nh1JRMVlmnBsB4IIsS91eG8Gs23prGoHC8fmv%2F%2B

 pip install snowflake-connector-python[secure-local-storage]


Loading data...
Aggregating OSA scores...
  Stores to process: 122
  Generating insights for store 497...
  Generating insights for store 536...
  Generating insights for store 637...
  Generating insights for store 759...
  Generating insights for store 474...
  Generating insights for store 475...
  Generating insights for store 492...
  Generating insights for store 457...
  Generating insights for store 461...
  Generating insights for store 205...
  Generating insights for store 217...
  Generating insights for store 239...
  Generating insights for store 406...
  Generating insights for store 408...
  Generating insights for store 423...
  Generating insights for store 432...
  Generating insights for store 433...
  Generating insights for store 506...
  Generating insights for store 508...
  Generating insights for store 517...
  Generating insights for store 520...
  Generating insights for store 530...
  Generating insights for store 631...
  Generating insights for store 764.